# **Estructura estándar recomendada (para todos los stage_XX_*.py)**

### **0) Docstring del módulo (contrato del stage)**

- Nombre del stage
- Propósito (1–2 líneas)
- Inputs / Outputs / Reports / Params
- Notas (supuestos y no-objetivos)

### **1) Imports**
- stdlib
- third-party
- local imports
- imports opcionales dentro de funciones (MLflow, etc.)

### **2) Logging (uniforme)**

- LOG_LEVEL por env
- log = logging.getLogger("stage_XX")

### **3) Configuración DVC-friendly (SIEMPRE presente)**

Siempre definir:

- IN_* (deps)
- OUT_* (outs)
- REPORT_* (reports)
- PARAMS (defaults reproducibles)

Ejemplo: 

```python
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
IN_RAW_PARQUET  = Path(os.environ.get("IN_RAW_PARQUET", "data/raw/mnq_raw.parquet"))
OUT_PARQUET     = Path(os.environ.get("OUT_PARQUET", "data/processed/mnq_intraday.parquet"))
REPORT_SUMMARY  = Path(os.environ.get("REPORT_SUMMARY", "reports/stage_01_dataset_prep_summary.json"))
```

Esto evita que cada stage “invente” paths via argparse y quede inconsistente.


### **4) Configuración funcional (parámetros del stage)**

Bloque “Params” con defaults:

```python
MARKET = os.environ.get("MARKET", "NASDAQ")
TZ_FROM = os.environ.get("TZ_FROM", "UTC")
TZ_TO = os.environ.get("TZ_TO", "America/New_York")
TRADING_START = os.environ.get("TRADING_START", "06:30:00")
TRADING_END = os.environ.get("TRADING_END", "16:00:00")
GAP_MINUTES = int(os.environ.get("GAP_MINUTES", "1"))
```

### **5) Utilidades puras (funciones “core”)**

- Sin side-effects (no escribir archivos dentro)
- Inputs/outputs claros
- Tipado + docstrings

### **6) Summary/Report**

```python
- build_stage_XX_summary(...) -> dict
- save_json(summary, REPORT_SUMMARY)
- print_stage_XX_summary_console(summary)
```
Envelope común (siempre presente):
```json
{
  "stage": "stage_01_intraday_data_preparation",
  "created_at_utc": "...",
  "version": "1.0",
  "paths": { "inputs": {...}, "outputs": {...}, "reports": {...} },
  "params": { ... },          // config del stage (normalizado)
  "metrics": { ... },         // métricas numéricas (normalizado)
  "details": { ... }          // libre, específico del stage
}
```

Claves:

- `params` y `metrics`: estructura estable (aunque cambien las claves internas)
- `details`: cada stage pone lo que necesite (tablas, listas, breakdowns por día/archivo, etc.)




### **7) Tracking MLflow (params/metrics + artifacts)**

- `--enable-mlflow` o `ENABLE_MLFLOW`
- import perezoso dentro de la función

- `mlflow_tracking(summary, *, enable=..., run_name=..., tags=..., artifacts=[REPORT_SUMMARY, ...])

Regla clave:

- El report JSON siempre se genera.
- MLflow es opcional, y si está activo:
    - loguea params (config)
    - loguea metrics (números)
    - loguea artifacts (el JSON del report + opcionalmente muestras/plots)


#### Convención para params y metrics (MLflow-friendly)

- params (configuración)
- Solo cosas “de control” y reproducibles: horarios, tz, umbrales, horizon, etc.
- Tipos simples (str/int/float/bool)

- metrics (números comparables)
- Solo valores numéricos finales (int/float)
- Nombres estables por stage (no hace falta que coincidan entre stages)

Ejemplo stage_01:

```json
"params": {
  "market": "NASDAQ",
  "trading_start": "06:30:00",
  "trading_end": "16:00:00",
  "tz_from": "UTC",
  "tz_to": "America/New_York",
  "gap_minutes": 1
},
"metrics": {
  "total_days_raw": 1200,
  "trading_days_output": 1187,
  "discarded_days": 13,
  "discarded_days_pct": 1.0833,
  "records_per_day_median": 451,
  "total_nans": 0
},
"details": {
  "records_per_day": { "min": 450, "median": 451, "max": 451 },
  "trading_time_range_effective": { "start": "...", "end": "..." },
  "days_with_gaps": 0
}
```


#### **8) parse_args() (opcional, pero estandarizado)**

Mi recomendación:
- Mantener argparse solo como override
- Defaults deben venir de los constantes DVC-friendly definidos arriba

Ejemplo:
```python
parser.add_argument("--in-raw", default=str(IN_RAW_PARQUET))
```

**9) main() (orquestación)**

- logging por pasos `[1]`, `[2]`, etc.
- no lógica pesada inline: usar funciones

10) Boilerplate

- `if __name__ == "__main__": main()`

In [ ]:
def mlflow_log_from_summary(*, enable: bool, stage: str, summary: dict, summary_path: Path) -> None:
    if not enable:
        return
    try:
        import mlflow
    except ImportError as exc:
        raise ImportError("MLflow no está instalado. Instale con: pip install mlflow") from exc

    with mlflow.start_run(run_name=stage):
        mlflow.set_tag("stage", stage)

        for k, v in (summary.get("params", {}) or {}).items():
            mlflow.log_param(k, v)

        for k, v in (summary.get("metrics", {}) or {}).items():
            if isinstance(v, (int, float)) and v == v:  # evita NaN
                mlflow.log_metric(k, float(v))

        mlflow.log_artifact(str(summary_path))